# NOC Prediction — Set Transformer — 10-Fold CV (All-in-One)

Notebook chay tu dau den cuoi:
1. Doc raw CSV tu PROVEDIt GF29cycles
2. Token hoa peak STR → `{(locus_idx, allele_val, log1p_height)}`
3. Train Set Transformer voi Stratified 10-Fold CV
4. Tong hop ket qua

Cau hinh theo bao cao: ISAB x2, PMA, d_model=128, n_heads=4, m=32,
locus_emb=16, dropout=0.1, AdamW lr=3e-4, ReduceLROnPlateau, early stop patience=15.

## 1. Setup & Imports

In [4]:
import subprocess
import sys
from pathlib import Path

PKGS = ["pandas", "scikit-learn", "tqdm"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PKGS], check=True)

VENDOR_ROOT = Path("/kaggle/working/set_transformer")
if not VENDOR_ROOT.exists():
    VENDOR_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "https://github.com/juho-lee/set_transformer.git", str(VENDOR_ROOT)],
        check=True,
    )
if str(VENDOR_ROOT) not in sys.path:
    sys.path.insert(0, str(VENDOR_ROOT))

import copy
import gzip
import json
import math
import os
import random
import re
import time
from collections import Counter, defaultdict
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report,
)
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset, Sampler
from tqdm.auto import tqdm

from modules import ISAB, PMA

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

Torch: 2.10.0+cu128
CUDA: True


Cloning into '/kaggle/working/set_transformer'...


## 2. Config

In [5]:
PROJECT_ROOT = Path('/kaggle/input/datasets/cquangnguynl/noc-unfiltered')
DATASET_ROOT = PROJECT_ROOT / "PROVEDIt_1-5-Person CSVs UnFiltered" / "PROVEDIt_1-5-Person CSVs UnFiltered_3500_GF29cycles"
OUTPUT_DIR =  Path("/kaggle/working/noc_10fold")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Data processing ---
MAX_LEN = 160

N_FOLDS = 10
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15
LR = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
SEED = 42
USE_AMP = True

# --- Model (theo bao cao slide 25) ---
LOCUS_EMB_DIM = 16
DIM_HIDDEN = 128
NUM_HEADS = 4
NUM_INDS = 32
DROPOUT = 0.1
NOC_NUM_CLASSES = 5

# --- LR scheduler (slide 26) ---
LR_SCHEDULER_FACTOR = 0.5
LR_SCHEDULER_PATIENCE = 5

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("DEVICE:", DEVICE)

PROJECT_ROOT: /kaggle/input/datasets/cquangnguynl/noc-unfiltered
DATASET_ROOT: /kaggle/input/datasets/cquangnguynl/noc-unfiltered/PROVEDIt_1-5-Person CSVs UnFiltered/PROVEDIt_1-5-Person CSVs UnFiltered_3500_GF29cycles
OUTPUT_DIR: /kaggle/working/noc_10fold
DEVICE: cuda


## 3. Data Processing — Raw CSV → Tokens

Doc truc tiep raw CSV tu PROVEDIt GF29cycles.
Moi allele peak (non-OL, height > 0) → 1 token: `(locus_idx, allele_val, log1p_height)`.
Max 160 tokens/sample (giu top height).

In [6]:
LOCUS_ORDER = [
    "D3S1358", "vWA", "D16S539", "CSF1PO", "TPOX", "Yindel", "AMEL",
    "D8S1179", "D21S11", "D18S51", "DYS391", "D2S441", "D19S433", "TH01",
    "FGA", "D22S1045", "D5S818", "D13S317", "D7S820", "SE33",
    "D10S1248", "D1S1656", "D12S391", "D2S1338",
]
LOCUS_TO_IDX = {name: idx for idx, name in enumerate(LOCUS_ORDER)}
NUM_LOCI = len(LOCUS_ORDER)

FOLDER_RE = re.compile(r"(?P<person_group>[1-5]-Person)/(?P<inj_sec>\d+)\ssec/")


def clean_value(text) -> str:
    return str(text).strip() if text is not None else ""


def parse_height(text: str):
    text = clean_value(text)
    if not text:
        return None
    try:
        return float(text)
    except ValueError:
        return None


def parse_allele_value(allele_raw: str):
    allele_raw = clean_value(allele_raw)
    if not allele_raw:
        return None
    if allele_raw == "X":
        return -2.0
    if allele_raw == "Y":
        return -1.0
    try:
        return float(allele_raw)
    except ValueError:
        return None


def build_allele_triplets(columns):
    triplets = []
    for col in columns:
        m = re.fullmatch(r"Allele (\d+)", col)
        if m:
            idx = m.group(1)
            triplets.append((col, f"Size {idx}", f"Height {idx}"))
    return triplets


def process_csv(csv_path: Path, max_len: int):
    rel = csv_path.as_posix()
    match = FOLDER_RE.search(rel)
    if not match:
        return []
    true_noc = int(match.group("person_group")[0])
    inj_sec = int(match.group("inj_sec"))

    df = pd.read_csv(csv_path, dtype=str, keep_default_na=False, na_filter=False, low_memory=False)
    triplets = build_allele_triplets(df.columns.tolist())
    records = []

    for sample_file, group in df.groupby("Sample File", sort=False):
        tokens = []
        for row in group.to_dict(orient="records"):
            marker = clean_value(row.get("Marker"))
            locus_idx = LOCUS_TO_IDX.get(marker)
            if locus_idx is None:
                continue
            for allele_col, size_col, height_col in triplets:
                allele_raw = clean_value(row.get(allele_col))
                if not allele_raw or allele_raw == "OL":
                    continue
                height = parse_height(row.get(height_col, ""))
                if height is None or height <= 0:
                    continue
                allele_val = parse_allele_value(allele_raw)
                if allele_val is None:
                    continue
                tokens.append({
                    "locus_idx": locus_idx,
                    "allele_val": allele_val,
                    "log1p_height": math.log1p(height),
                    "height": height,
                })

        if not tokens:
            continue

        tokens.sort(key=lambda t: t["height"], reverse=True)
        selected = tokens[:max_len]
        selected.sort(key=lambda t: (t["locus_idx"], t["allele_val"], -t["height"]))

        records.append({
            "sample_uid": f"{csv_path.name}::{sample_file}",
            "true_noc": true_noc,
            "injection_time_sec": inj_sec,
            "token_locus_idx": [t["locus_idx"] for t in selected],
            "token_allele_val": [t["allele_val"] for t in selected],
            "token_log1p_height": [t["log1p_height"] for t in selected],
            "seq_len": len(selected),
        })
    return records


# --- Build dataset ---
csv_paths = sorted(DATASET_ROOT.glob("*-Person/* sec/*.csv"))
print(f"Found {len(csv_paths)} CSV files")

all_records = []
for csv_path in tqdm(csv_paths, desc="Processing CSVs"):
    all_records.extend(process_csv(csv_path, MAX_LEN))

labels = np.array([r["true_noc"] for r in all_records])
noc_dist = Counter(labels)
print(f"\nTotal samples: {len(all_records)}")
print(f"NOC distribution: {dict(sorted(noc_dist.items()))}")
print(f"Num loci: {NUM_LOCI}, Max len: {MAX_LEN}")

Found 183 CSV files


Processing CSVs:   0%|          | 0/183 [00:00<?, ?it/s]


Total samples: 10195
NOC distribution: {np.int64(1): 8190, np.int64(2): 526, np.int64(3): 484, np.int64(4): 527, np.int64(5): 468}
Num loci: 24, Max len: 160


## 4. Dataset & DataLoader

In [7]:
def batch_size_for_length(seq_len: int) -> int:
    if seq_len <= 64:
        return 128
    if seq_len <= 96:
        return 96
    if seq_len <= 128:
        return 64
    return 48


class NOCDataset(Dataset):
    def __init__(self, records, scalar_stats):
        self.records = records
        self.ss = scalar_stats

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        allele = (np.asarray(r["token_allele_val"], dtype=np.float32) - self.ss["allele_mean"]) / self.ss["allele_std"]
        log_h = (np.asarray(r["token_log1p_height"], dtype=np.float32) - self.ss["log_height_mean"]) / self.ss["log_height_std"]
        return {
            "locus_idx": torch.tensor(r["token_locus_idx"], dtype=torch.long),
            "allele_val": torch.tensor(allele, dtype=torch.float32),
            "log1p_height": torch.tensor(log_h, dtype=torch.float32),
            "noc_target": torch.tensor(r["true_noc"] - 1, dtype=torch.long),
            "seq_len": len(r["token_locus_idx"]),
        }


class ExactLengthBatchSampler(Sampler):
    def __init__(self, records, shuffle: bool, seed: int):
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0
        self.by_len = defaultdict(list)
        for idx, r in enumerate(records):
            self.by_len[len(r["token_locus_idx"])].append(idx)

    def set_epoch(self, epoch: int):
        self.epoch = epoch

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)
        lengths = list(self.by_len.keys())
        if self.shuffle:
            rng.shuffle(lengths)
        batches = []
        for length in lengths:
            indices = list(self.by_len[length])
            if self.shuffle:
                rng.shuffle(indices)
            bs = batch_size_for_length(length)
            for start in range(0, len(indices), bs):
                batches.append(indices[start:start + bs])
        if self.shuffle:
            rng.shuffle(batches)
        return iter(batches)

    def __len__(self):
        total = 0
        for length, indices in self.by_len.items():
            bs = batch_size_for_length(length)
            total += math.ceil(len(indices) / bs)
        return total


def collate_fn(batch):
    return {
        "locus_idx": torch.stack([b["locus_idx"] for b in batch]),
        "allele_val": torch.stack([b["allele_val"] for b in batch]),
        "log1p_height": torch.stack([b["log1p_height"] for b in batch]),
        "noc_target": torch.stack([b["noc_target"] for b in batch]),
        "seq_len": batch[0]["seq_len"],
    }


def compute_scalar_stats(records):
    allele_vals, log_heights = [], []
    for r in records:
        allele_vals.extend(r["token_allele_val"])
        log_heights.extend(r["token_log1p_height"])
    allele_vals = np.asarray(allele_vals, dtype=np.float32)
    log_heights = np.asarray(log_heights, dtype=np.float32)
    return {
        "allele_mean": float(allele_vals.mean()),
        "allele_std": float(max(allele_vals.std(), 1e-6)),
        "log_height_mean": float(log_heights.mean()),
        "log_height_std": float(max(log_heights.std(), 1e-6)),
    }


def make_loader(records, scalar_stats, shuffle: bool, seed: int):
    ds = NOCDataset(records, scalar_stats)
    sampler = ExactLengthBatchSampler(records, shuffle=shuffle, seed=seed)
    return DataLoader(ds, batch_sampler=sampler, collate_fn=collate_fn, num_workers=0), sampler

print("Dataset & loader utilities ready.")

Dataset & loader utilities ready.


## 5. Model

- Locus Embedding(24, 16) → input dim = 18
- ISAB(128, 4 heads, m=32, ln=True) x 2
- PMA(128, 4 heads, 1 seed, ln=True) → z_mix (128-dim)
- Dropout(0.1)
- Linear(128, 5)

In [8]:
class SetTransformerNOC(nn.Module):
    def __init__(self, num_loci, noc_num_classes=5,
                 locus_emb_dim=16, dim_hidden=128,
                 num_heads=4, num_inds=32, dropout=0.1):
        super().__init__()
        self.locus_emb = nn.Embedding(num_loci, locus_emb_dim)
        dim_input = locus_emb_dim + 2
        self.enc = nn.Sequential(
            ISAB(dim_input, dim_hidden, num_heads, num_inds, ln=True),
            ISAB(dim_hidden, dim_hidden, num_heads, num_inds, ln=True),
        )
        self.pool = PMA(dim_hidden, num_heads, 1, ln=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(dim_hidden, noc_num_classes)

    def forward(self, locus_idx, allele_val, log1p_height):
        emb = self.locus_emb(locus_idx)
        x = torch.cat([emb, torch.stack([allele_val, log1p_height], dim=-1)], dim=-1)
        h = self.enc(x)
        z = self.pool(h).squeeze(1)
        z = self.dropout(z)
        return self.head(z)

_test = SetTransformerNOC(NUM_LOCI, dropout=DROPOUT)
print(f"Model params: {sum(p.numel() for p in _test.parameters()):,}")
del _test

Model params: 299,909


## 6. Training Utilities

In [9]:
def compute_noc_class_weights(records, num_classes):
    counts = np.bincount(
        [r["true_noc"] - 1 for r in records], minlength=num_classes
    ).astype(np.float32)
    weights = counts.max() / np.clip(counts, 1.0, None)
    return torch.tensor(weights, dtype=torch.float32)


def train_one_epoch(model, loader, sampler, optimizer, scaler, criterion, epoch):
    model.train()
    sampler.set_epoch(epoch)
    running_loss = 0.0
    steps = 0
    for batch in loader:
        batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == "cuda"):
            logits = model(batch["locus_idx"], batch["allele_val"], batch["log1p_height"])
            loss = criterion(logits, batch["noc_target"])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        steps += 1
    return running_loss / max(steps, 1)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    all_preds, all_targets = [], []
    running_loss = 0.0
    steps = 0
    for batch in loader:
        batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        logits = model(batch["locus_idx"], batch["allele_val"], batch["log1p_height"])
        loss = criterion(logits, batch["noc_target"])
        running_loss += loss.item()
        steps += 1
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(batch["noc_target"].cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    avg_loss = running_loss / max(steps, 1)
    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
    cm = confusion_matrix(all_targets, all_preds, labels=list(range(NOC_NUM_CLASSES)))
    return {
        "loss": avg_loss,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "confusion_matrix": cm.tolist(),
        "preds": all_preds,
        "targets": all_targets,
    }

print("Training utilities ready.")

Training utilities ready.


## 7. Train — 10-Fold Cross-Validation

- StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
- Early stopping theo **val Macro F1** (patience=15)
- ReduceLROnPlateau x0.5 (patience=5) theo val loss
- Luu checkpoint moi fold

In [12]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_results = []

for fold_idx, (train_indices, val_indices) in enumerate(skf.split(all_records, labels)):
    print(f"\n{'='*60}")
    print(f"FOLD {fold_idx + 1}/{N_FOLDS}")
    print(f"{'='*60}")

    train_records = [all_records[i] for i in train_indices]
    val_records = [all_records[i] for i in val_indices]

    train_noc = Counter(r["true_noc"] for r in train_records)
    val_noc = Counter(r["true_noc"] for r in val_records)
    print(f"Train: {len(train_records)} samples, NOC dist: {dict(sorted(train_noc.items()))}")
    print(f"Val:   {len(val_records)} samples, NOC dist: {dict(sorted(val_noc.items()))}")

    scalar_stats = compute_scalar_stats(train_records)
    train_loader, train_sampler = make_loader(train_records, scalar_stats, shuffle=True, seed=SEED + fold_idx)
    val_loader, _ = make_loader(val_records, scalar_stats, shuffle=False, seed=SEED)

    seed_everything(SEED + fold_idx)
    model = SetTransformerNOC(
        num_loci=NUM_LOCI,
        noc_num_classes=NOC_NUM_CLASSES,
        locus_emb_dim=LOCUS_EMB_DIM,
        dim_hidden=DIM_HIDDEN,
        num_heads=NUM_HEADS,
        num_inds=NUM_INDS,
        dropout=DROPOUT,
    ).to(DEVICE)

    class_weights = compute_noc_class_weights(train_records, NOC_NUM_CLASSES).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=LR_SCHEDULER_FACTOR,
        patience=LR_SCHEDULER_PATIENCE,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and DEVICE.type == "cuda")

    best_val_f1 = -1.0
    best_state = None
    best_epoch = -1
    patience = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, train_sampler, optimizer, scaler, criterion, epoch)
        val_result = evaluate(model, val_loader, criterion)
        scheduler.step(val_result["loss"])
        elapsed = time.time() - t0

        current_lr = optimizer.param_groups[0]["lr"]
        if epoch % 5 == 1 or epoch == MAX_EPOCHS:
            print(f"  Epoch {epoch:3d} | train_loss={train_loss:.4f} | "
                  f"val_loss={val_result['loss']:.4f} | "
                  f"val_acc={val_result['accuracy']:.4f} | "
                  f"val_f1={val_result['macro_f1']:.4f} | "
                  f"lr={current_lr:.2e} | {elapsed:.1f}s")

        if val_result["macro_f1"] > best_val_f1:
            best_val_f1 = val_result["macro_f1"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience = 0
        else:
            patience += 1
            if patience >= EARLY_STOPPING_PATIENCE:
                print(f"  Early stopping at epoch {epoch} (best val F1={best_val_f1:.4f} at epoch {best_epoch})")
                break

    model.load_state_dict(best_state)
    final_val = evaluate(model, val_loader, criterion)
    print(f"  Best epoch: {best_epoch} | val_acc={final_val['accuracy']:.4f} | val_f1={final_val['macro_f1']:.4f}")
    print(f"  Confusion matrix:\n{np.array(final_val['confusion_matrix'])}")

    fold_result = {
        "fold": fold_idx + 1,
        "best_epoch": best_epoch,
        "val_loss": final_val["loss"],
        "val_accuracy": final_val["accuracy"],
        "val_macro_f1": final_val["macro_f1"],
        "confusion_matrix": final_val["confusion_matrix"],
        "val_preds": final_val["preds"].tolist(),
        "val_targets": final_val["targets"].tolist(),
        "val_indices": val_indices.tolist(),
    }
    fold_results.append(fold_result)

    ckpt_path = OUTPUT_DIR / f"fold{fold_idx+1}_best.pt"
    torch.save({
        "model_state_dict": best_state,
        "fold": fold_idx + 1,
        "best_epoch": best_epoch,
        "scalar_stats": scalar_stats,
        "config": {
            "num_loci": NUM_LOCI,
            "noc_num_classes": NOC_NUM_CLASSES,
            "locus_emb_dim": LOCUS_EMB_DIM,
            "dim_hidden": DIM_HIDDEN,
            "num_heads": NUM_HEADS,
            "num_inds": NUM_INDS,
            "dropout": DROPOUT,
        },
    }, ckpt_path)

print(f"\nAll fold checkpoints saved to {OUTPUT_DIR}")


FOLD 1/10
Train: 9175 samples, NOC dist: {1: 7371, 2: 473, 3: 436, 4: 474, 5: 421}
Val:   1020 samples, NOC dist: {1: 819, 2: 53, 3: 48, 4: 53, 5: 47}


/tmp/ipykernel_58/3095063248.py:39: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and DEVICE.type == "cuda")
/tmp/ipykernel_58/1915559079.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == "cuda"):


  Epoch   1 | train_loss=1.4698 | val_loss=1.4041 | val_acc=0.4029 | val_f1=0.2024 | lr=3.00e-04 | 5.7s


/tmp/ipykernel_58/1915559079.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == "cuda"):
/tmp/ipykernel_58/1915559079.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == "cuda"):
/tmp/ipykernel_58/1915559079.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == "cuda"):
/tmp/ipykernel_58/1915559079.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == "cuda"):
/tmp/ipykernel_58/1915559079.py:17: FutureWarning: `torch.cuda.a

  Epoch   6 | train_loss=1.2610 | val_loss=1.2755 | val_acc=0.3069 | val_f1=0.1552 | lr=3.00e-04 | 4.3s


...

  Epoch  61 | train_loss=0.2228 | val_loss=0.6785 | val_acc=0.9422 | val_f1=0.8088 | lr=9.37e-06 | 4.3s


/tmp/ipykernel_58/1915559079.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == "cuda"):
/tmp/ipykernel_58/1915559079.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == "cuda"):
/tmp/ipykernel_58/1915559079.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == "cuda"):
/tmp/ipykernel_58/1915559079.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == "cuda"):
/tmp/ipykernel_58/1915559079.py:17: FutureWarning: `torch.cuda.a

  Epoch  66 | train_loss=0.2308 | val_loss=0.6827 | val_acc=0.9412 | val_f1=0.8044 | lr=4.69e-06 | 4.3s


/tmp/ipykernel_58/1915559079.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == "cuda"):
/tmp/ipykernel_58/1915559079.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == "cuda"):
/tmp/ipykernel_58/1915559079.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == "cuda"):


KeyboardInterrupt: 

## 8. Aggregate Results

In [ ]:
accs = [r["val_accuracy"] for r in fold_results]
f1s = [r["val_macro_f1"] for r in fold_results]

print("=" * 60)
print("10-FOLD CROSS-VALIDATION SUMMARY")
print("=" * 60)
for r in fold_results:
    print(f"  Fold {r['fold']:2d}: acc={r['val_accuracy']:.4f}  macro_f1={r['val_macro_f1']:.4f}  best_epoch={r['best_epoch']}")
print("-" * 60)
print(f"  Mean acc:      {np.mean(accs):.4f} +/- {np.std(accs):.4f}")
print(f"  Mean macro_f1: {np.mean(f1s):.4f} +/- {np.std(f1s):.4f}")

all_preds = []
all_targets = []
for r in fold_results:
    all_preds.extend(r["val_preds"])
    all_targets.extend(r["val_targets"])

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

print("\nAggregated confusion matrix (all folds):")
agg_cm = confusion_matrix(all_targets, all_preds, labels=list(range(NOC_NUM_CLASSES)))
noc_labels = ["1-Person", "2-Person", "3-Person", "4-Person", "5-Person"]
print(f"{'':>10s}", "  ".join(f"{l:>9s}" for l in noc_labels))
for i, row in enumerate(agg_cm):
    print(f"{noc_labels[i]:>10s}", "  ".join(f"{v:9d}" for v in row))

print("\nPer-class report:")
print(classification_report(
    all_targets, all_preds,
    target_names=noc_labels, digits=4, zero_division=0,
))

summary = {
    "n_folds": N_FOLDS,
    "total_samples": len(all_records),
    "noc_distribution": {str(k): int(v) for k, v in sorted(noc_dist.items())},
    "mean_accuracy": float(np.mean(accs)),
    "std_accuracy": float(np.std(accs)),
    "mean_macro_f1": float(np.mean(f1s)),
    "std_macro_f1": float(np.std(f1s)),
    "config": {
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "max_epochs": MAX_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "lr_scheduler_factor": LR_SCHEDULER_FACTOR,
        "lr_scheduler_patience": LR_SCHEDULER_PATIENCE,
        "dim_hidden": DIM_HIDDEN,
        "num_heads": NUM_HEADS,
        "num_inds": NUM_INDS,
        "locus_emb_dim": LOCUS_EMB_DIM,
        "dropout": DROPOUT,
        "noc_num_classes": NOC_NUM_CLASSES,
        "max_len": MAX_LEN,
    },
    "per_fold": [{
        "fold": r["fold"],
        "accuracy": r["val_accuracy"],
        "macro_f1": r["val_macro_f1"],
        "best_epoch": r["best_epoch"],
        "confusion_matrix": r["confusion_matrix"],
    } for r in fold_results],
    "aggregated_confusion_matrix": agg_cm.tolist(),
}
summary_path = OUTPUT_DIR / "cv_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"\nSaved summary to {summary_path}")